# 6 Fine-tuning for classification

## 6.2 Preparing the dataset

In [1]:
import urllib.request
import zipfile
import os
from pathlib import Path

import pandas as pd

In [2]:
url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path = "sms_spam_collection.zip"
extracted_path = "sms_spam_collection"
data_file_path = Path(extracted_path) / "SMSSpamCollection.tsv"

def download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path):
    if data_file_path.exists():
        print(f"{data_file_path} already exists. Skipping download and extration.")
        return
    
    # 下载zip文件
    with urllib.request.urlopen(url) as response:
        with open(zip_path, "wb") as out_file:
            out_file.write(response.read())
    # unzip
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extracted_path)

    original_file_path = Path(extracted_path) / "SMSSpamCollection"
    os.rename(original_file_path, data_file_path) # 添加.tsv 后缀
    print(f"File downloaded and saved as {data_file_path}")

download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)

File downloaded and saved as sms_spam_collection/SMSSpamCollection.tsv


In [3]:
# 文件使用 tab 分割
df = pd.read_csv(data_file_path, sep="\t", header=None, names=["Label", "Text"])
df

,Label,Text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [4]:
print(df["Label"].value_counts())   # 

Label
ham     4825
spam     747
Name: count, dtype: int64


大部分都是ham，后面需要匹配数量

In [5]:
print(df["Label"]=="spam")      # 判断每行是否是spam
print(df[df["Label"]=="spam"])  # 筛选所有spam行

0       False
1       False
2        True
3       False
4       False
        ...  
5567     True
5568    False
5569    False
5570    False
5571    False
Name: Label, Length: 5572, dtype: bool
     Label                                               Text
2     spam  Free entry in 2 a wkly comp to win FA Cup fina...
5     spam  FreeMsg Hey there darling it's been 3 week's n...
8     spam  WINNER!! As a valued network customer you have...
9     spam  Had your mobile 11 months or more? U R entitle...
11    spam  SIX chances to win CASH! From 100 to 20,000 po...
...    ...                                                ...
5537  spam  Want explicit SEX in 30 secs? Ring 02073162414...
5540  spam  ASKED 3MOBILE IF 0870 CHATLINES INCLU IN FREE ...
5547  spam  Had your contract mobile 11 Mnths? Latest Moto...
5566  spam  REMINDER FROM O2: To get 2.50 pounds free call...
5567  spam  This is the 2nd time we have tried 2 contact u...

[747 rows x 2 columns]


In [6]:
def create_balanced_dataset(df):
    num_spam = df[df["Label"] == "spam"].shape[0]
    print(num_spam)
    # 随机对ham抽样，匹配spam的个数
    ham_subset = df[df["Label"] == "ham"].sample(num_spam, random_state=123)
    balanced_df = pd.concat([ham_subset, df[df["Label"] == "spam"]])
    return balanced_df

In [7]:
balanced_df = create_balanced_dataset(df)
print(balanced_df["Label"].value_counts())

747
Label
ham     747
spam    747
Name: count, dtype: int64


In [8]:
# map 前
create_balanced_dataset(df)["Label"][:3]

747


,Label
4307,ham
4138,ham
4831,ham


In [9]:
balanced_df["Label"] = balanced_df["Label"].map({"ham": 0, "spam": 1})

In [10]:
print(len(balanced_df))
# 抽样所有，相当于打乱
shuffled = balanced_df.sample(frac=1)
print(shuffled[:3])
# 重置下标
shuffled = shuffled.reset_index(drop=True)  # drop=True 去除旧的下标列
print(shuffled[:3])

print(len(shuffled))

1494
      Label                                               Text
2297      1  <Forwarded from 21870000>Hi - this is your Mai...
2148      0  Ok. Can be later showing around 8-8:30 if you ...
474       1  Want 2 get laid tonight? Want real Dogging loc...
   Label                                               Text
0      1  <Forwarded from 21870000>Hi - this is your Mai...
1      0  Ok. Can be later showing around 8-8:30 if you ...
2      1  Want 2 get laid tonight? Want real Dogging loc...
1494


In [11]:
def random_split(df, train_frac, validation_frac):
    # 分为 train, valid, test集，
    df = df.sample(frac=1, random_state=123).reset_index(drop=True)
    train_end = int(len(df) * train_frac)
    validation_end = train_end + int(len(df) * validation_frac)

    train_df = df[: train_end]
    validation_df = df[train_end:validation_end]
    test_df = df[validation_end:]

    return train_df, validation_df, test_df

train_df, validation_df, test_df = random_split(balanced_df, 0.7, 0.1)

In [12]:
train_df. to_csv("train.csv", index=None)
validation_df.to_csv("validation.csv", index=None)
test_df.to_csv("test.csv", index=None)

In [13]:
train_df

,Label,Text
0,0,Dude how do you like the buff wind.
1,0,Tessy..pls do me a favor. Pls convey my birthd...
2,1,Reminder: You have not downloaded the content ...
3,1,Got what it takes 2 take part in the WRC Rally...
4,1,"Shop till u Drop, IS IT YOU, either 10K, 5K, £..."
...,...,...
1040,1,4mths half price Orange line rental & latest c...
1041,1,Thanks for the Vote. Now sing along with the s...
1042,1,IMPORTANT INFORMATION 4 ORANGE USER 0796XXXXXX...
1043,1,Urgent! call 09066612661 from landline. Your c...


## 6.3 Creating data loaders

![text to padded token ids](https://raw.githubusercontent.com/ipdor/Pictures/master/20260502113416459.png)

In [14]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
print(tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"}))

[50256]


在实例化数据加载器之前，我们首先需要实现一个 PyTorch Dataset，它指定了如何加载和处理数据


In [15]:
tks = [22, 21, 20]
pad = [31] * 3
print(tks + pad)

[22, 21, 20, 31, 31, 31]


In [16]:
data = pd.read_csv("train.csv")
print(data["Label"])   # 输出列
print(data[0:5])       # 输出行
# 使用 iloc 输出第一行
print(data.iloc[0])

0       0
1       0
2       1
3       1
4       1
       ..
1040    1
1041    1
1042    1
1043    1
1044    0
Name: Label, Length: 1045, dtype: int64
   Label                                               Text
0      0                Dude how do you like the buff wind.
1      0  Tessy..pls do me a favor. Pls convey my birthd...
2      1  Reminder: You have not downloaded the content ...
3      1  Got what it takes 2 take part in the WRC Rally...
4      1  Shop till u Drop, IS IT YOU, either 10K, 5K, £...
Label                                      0
Text     Dude how do you like the buff wind.
Name: 0, dtype: object


In [23]:
max_lens = max([len(x) for x in data["Text"]])
max_lens

458

In [39]:

tokens = [tokenizer.encode(x) for x in data["Text"]]
max_token_lens = max([len(token) for token in tokens])
print(max_token_lens)

token = tokens[0]
print(token)
pad = 27
padded_token = token +  [pad] * (max_token_lens - len(token))
print(len(padded_token))
print(padded_token)

120
[35, 2507, 703, 466, 345, 588, 262, 6940, 2344, 13]
120
[35, 2507, 703, 466, 345, 588, 262, 6940, 2344, 13, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27]


In [79]:
import torch
from torch.utils.data import Dataset

class SpamDataset(Dataset):
    def __init__(self, csv_file, tokenizer, max_length=None, pad_token_id=5026):
        self.data = pd.read_csv(csv_file)
        # 预处理，各段text转换为token id
        self.encoded_texts = [tokenizer.encode(x) for x in data["Text"]]

        if max_length is None:
            self.max_length = self.__longest_encoded_length()
        else:
            self.max_length = max_length
            # 超过最大长度就截断
            self.encoded_texts = [encoded_text[:self.max_length] for encoded_text in self.encoded_texts]
        # 将序列填充到最长序列的长度
        self.encoded_texts = [encoded_text + [pad_token_id] * (self.max_length - len(encoded_text)) for encoded_text in self.encoded_texts]
        
    # 返回 (token id, label) 元组
    def __getitem__(self, index):
        return (torch.tensor(self.encoded_texts[index], dtype=torch.long),
                torch.tensor(self.data["Label"][index], dtype=torch.long))

    # 返回 data 长度
    def __len__(self):
        return len(self.data)
    
    # 计算直接编码后得到的token id中最长的长度
    def __longest_encoded_length(self):
        max_token_lens = max([len(token) for token in self.encoded_texts])
        return max_token_lens

In [80]:
train_dataset = SpamDataset(csv_file="train.csv", 
                            max_length=None, 
                            tokenizer=tokenizer)

In [81]:
print(len(train_dataset))
print(train_dataset.max_length)


1045
120


In [82]:
val_dataset = SpamDataset(csv_file="validation.csv", 
                            max_length=None, 
                            tokenizer=tokenizer)

In [83]:
test_dataset = SpamDataset(csv_file="test.csv", 
                            max_length=None, 
                            tokenizer=tokenizer)

In [84]:
print(val_dataset.max_length)
print(test_dataset.max_length)

120
120


### Exercise 6.1 Increasing the context length

将输入填充到模型支持的最大 token 数量，并观察其如何影响预测性能

（预测训练会更慢

In [85]:
train_dataset_maxlen = SpamDataset(csv_file="train.csv", 
                            max_length=1024, 
                            tokenizer=tokenizer)

val_dataset_maxlen = SpamDataset(csv_file="validation.csv", 
                            max_length=1024, 
                            tokenizer=tokenizer)

test_dataset_maxlen = SpamDataset(csv_file="test.csv", 
                            max_length=1024, 
                            tokenizer=tokenizer)

print(train_dataset_maxlen.max_length)
print(valid_dataset_maxlen.max_length)
print(test_dataset_maxlen.max_length)


1024
1024
1024


In [86]:
from torch.utils.data import DataLoader

num_workers = 0     # 确保兼容性
batch_size = 8
torch.manual_seed(123)

train_loader = DataLoader(
    dataset = train_dataset,
    batch_size = batch_size,
    shuffle = True,
    num_workers = num_workers,
    drop_last = True
)

val_loader = DataLoader(
    dataset = val_dataset,
    batch_size = batch_size,
    num_workers = num_workers,
    drop_last = False
)

test_loader = DataLoader(
    dataset = test_dataset,
    batch_size = batch_size,
    num_workers = num_workers,
    drop_last = False
)

In [87]:
for input_batch, target_batch in train_loader:
    pass

print("Input batch dimensions:", input_batch.shape)
print("Label batch dimensions", target_batch.shape)

Input batch dimensions: torch.Size([8, 120])
Label batch dimensions torch.Size([8])


In [88]:
print(f"{len(train_loader)} training batches")
print(f"{len(val_loader)} validation batches")
print(f"{len(test_loader)} test batches")

130 training batches
19 validation batches
38 test batches


## 6.4 Initializing a model with pretrained weights